In [1]:
import sys
sys.path.insert(0, '../src')

%load_ext autoreload
%autoreload 2

In [2]:
from generator_multidim import MultidimSampleGenerator
from run_comparison import run_comparison

NUM_SAMPLES = 100
SUPP = 1.0
SAMPLE_KWARGS = dict(
    sample_size=6,
    min_trace_length=5,
    max_trace_length=10,
    event_dimension=1,
    type_count=5,
)

gen = MultidimSampleGenerator()

2026-06-16 15:03:17,592	INFO worker.py:2012 -- Started a local Ray instance.
C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [3]:
all_results = []   # list of (sample_id, results_list)
discrepancies = [] # list of (sample_id, algo_a, algo_b, set_a, set_b)

for i in range(NUM_SAMPLES):
    sample = gen.generate_random_sample(**SAMPLE_KWARGS)
    results = run_comparison(sample, SUPP)
    all_results.append((i, sample, results))

    # Check every pair of algorithms for differing querysets
    for j in range(len(results)):
        for k in range(j + 1, len(results)):
            a, b = results[j], results[k]
            if a['queryset'] != b['queryset']:
                discrepancies.append({
                    'sample_id': i,
                    'algo_a': a['algorithm'],
                    'algo_b': b['algorithm'],
                    'only_in_a': a['queryset'] - b['queryset'],
                    'only_in_b': b['queryset'] - a['queryset'],
                    'sample_traces': list(sample._sample),
                })

print(f"Tested {NUM_SAMPLES} samples.")
print(f"Discrepancies found: {len(discrepancies)}")

(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 628 in job_logging_config
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 2801 in disconnect
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 2132 in shutdown
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 1143 in wrapper
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\client_mode_hook.py", line 107 in wrapper


Tested 100 samples.
Discrepancies found: 0


(raylet) Stack (most recent call first): [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 628 in job_logging_config [repeated 3x across cluster]
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 2801 in disconnect [repeated 3x across cluster]
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 2132 in shutdown [repeated 3x across cluster]
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_improved\.venv\Lib\site-packages\ray\_private\worker.py", line 1143 in wrapper [repeated 3x across cluster]
(raylet)   File "C:\Users\Noel\PycharmProjects\disces_i

In [4]:
# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'Sample':>8}  {'Algorithm':<12}  {'#Queries':>8}  Queryset")
print("-" * 70)
for sample_id, sample, results in all_results:
    for r in results:
        print(f"{sample_id:>8}  {r['algorithm']:<12}  {r['queries_found']:>8}  {sorted(r['queryset'])}")
    print()

  Sample  Algorithm     #Queries  Queryset
----------------------------------------------------------------------
       0  D-U-C                1  ['$x0; $x0;']
       0  D-U-C-S              1  ['$x0; $x0;']
       0  D-U-S                1  ['$x0; $x0;']
       0  B-S-C                1  ['$x0; $x0;']
       0  B-S-S                1  ['$x0; $x0;']

       1  D-U-C                2  ['$x0; $x0;', 'ab;']
       1  D-U-C-S              2  ['$x0; $x0;', 'ab;']
       1  D-U-S                2  ['$x0; $x0;', 'ab;']
       1  B-S-C                2  ['$x0; $x0;', 'ab;']
       1  B-S-S                2  ['$x0; $x0;', 'ab;']

       2  D-U-C                1  ['$x0; $x0;']
       2  D-U-C-S              1  ['$x0; $x0;']
       2  D-U-S                1  ['$x0; $x0;']
       2  B-S-C                1  ['$x0; $x0;']
       2  B-S-S                1  ['$x0; $x0;']

       3  D-U-C                2  ['$x0; $x1; $x0; $x1;', 'ad;']
       3  D-U-C-S              2  ['$x0; $x1; $x0; $x1;', 'ad;'

In [5]:
# ── Discrepancy detail ───────────────────────────────────────────────────────
if not discrepancies:
    print("✅  All algorithms agree on every sample.")
else:
    for d in discrepancies:
        print(f"\n{'='*60}")
        print(f"Sample #{d['sample_id']}")
        print(f"  Traces: {d['sample_traces']}")
        print(f"  {d['algo_a']} vs {d['algo_b']}")
        print(f"  Only in {d['algo_a']}: {d['only_in_a']}")
        print(f"  Only in {d['algo_b']}: {d['only_in_b']}")

✅  All algorithms agree on every sample.
